In [ ]:
import copy
import networkx as nx
import pandas as pd
import statsmodels.api as sm
import sys
import warnings

from dowhy import CausalModel
from econml.inference import BootstrapInference
from pathlib import Path
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.ensemble import GradientBoostingRegressor
from word2num import Word2Num

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

from config_handlers.dataset_handler import DatasetHandler
from config_handlers.causal_handler import CausalHandler
from config_handlers.rk_handler import RKHandler
from config_handlers.sk_handler import SKHandler
from utils.data_io import load_jsonl, load_pickle, make_dir, save_json
# from utils.vis import save_graph_to_img

warnings.filterwarnings("ignore")

In [2]:
PROMPT_VERSION = 4
MODEL = "internvl2"
DATASET = "vqav2"

In [ ]:
rk_hdl = RKHandler()
rk_hdl.set_curr_model(MODEL)
rk_hdl.set_curr_ds(DATASET)
sk_hdl = SKHandler()
sk_hdl.set_curr_ds(DATASET)
ds_hdl = DatasetHandler()
ds_hdl.set_curr_ds(DATASET)

ds_list = ds_hdl.get_ds_list()

Load responses_counter
- These files already include the answer for the "complete RK"

In [ ]:
# TODO: change this to Path(__file__).parent for script
base_dir = Path("..")
resps_path = base_dir.joinpath(str(rk_hdl.get_rk_parsed_path()).format(PROMPT_VERSION))
print(resps_path)
resps = load_jsonl(resps_path)

alt_resps_path = base_dir.joinpath(rk_hdl.get_rk_counterfactual_path())
print(alt_resps_path)
alt_resps = load_jsonl(alt_resps_path)

../data/really_know/parsed/internvl2/vqav2/results_v4.jsonl
../data/really_know/counterfactual/internvl2/vqav2/results.jsonl


In [5]:
# Get responses only for the max num of triplets
# (i.e., ignore the rest of the powerset for now)
resps_ids = set([q["question_id"] for q in resps])
alt_resps_ids = set([q["question_id"] for q in alt_resps])

# question_id: response
dict_eval_resp = {
    q_idx: next(r["response"] for r in resps if r["question_id"] == q_idx)
    for q_idx in resps_ids
}

# question_id: response
# this only gets the first response
dict_counter_resp = {
    q_idx: next(r["response"] for r in alt_resps if r["question_id"] == q_idx)
    for q_idx in alt_resps_ids
}

For each question, add a new entry corresponding to the original response.

In [6]:
resps[0].keys()

dict_keys(['question_id', 'response', 'rationales', 'triples', 'triple_objs'])

In [7]:
# Based on the counterfactual responses we have, pick and add the original ones.
all_resps = copy.deepcopy(alt_resps)
for q_idx in alt_resps_ids:
    og_response = next((r for r in resps if r["question_id"] == q_idx), None)
    if og_response:
        all_resps.append(
            {
                "question_id": og_response["question_id"],
                "size_treatment": 0,
                "occluded_concepts": [],
                "response": og_response["response"],
            }
        )
    else:
        print(f"We have an issue with question {q_idx}.")

Create dataframe out of counterfactual data and responses

In [8]:
df = pd.DataFrame(all_resps)
df.drop(["path"], axis=1, inplace=True)

In [9]:
df.head()

,question_id,size_treatment,occluded_concepts,response
0,337814001,1,[28-table],3
1,337814001,1,[6-wall],3
2,337814001,1,[26-chair],3
3,337814001,1,[2-leg],3
4,337814001,1,[1-wall],3


Make ```response``` lowercase for consistency

In [10]:
df["response"] = df["response"].apply(lambda x: x.lower())

In [11]:
df[df["size_treatment"] == 0]

,question_id,size_treatment,occluded_concepts,response
338840,224000000,0,[],atv
338841,399744001,0,[],13
338842,544533000,0,[],0
338843,154520073,0,[],flying kite
338844,471893002,0,[],1
...,...,...,...,...
338985,229889007,0,[],pole
338986,337814001,0,[],3
338987,249046003,0,[],snowboarding
338988,291286004,0,[],skateboard


In [12]:
# Create mapping between questions and the biggest size of constraints.
dict_max_triplets = {
    q_idx: max(
        [int(q["size_treatment"]) for q in alt_resps if q["question_id"] == q_idx]
    )
    for q_idx in alt_resps_ids
}

# Make dictionary with {question_id: [concepts]}.
# This is done to extract an actual list of concepts, not any other PD objects.
dict_concepts = {
    q_idx: df[
        (df["question_id"] == q_idx)
        & (df["size_treatment"] == dict_max_triplets[q_idx])
    ]["occluded_concepts"].iloc[0]
    for q_idx in dict_max_triplets
}

Fix numerical answers given as words (e.g., "Three")

In [13]:
def fix_word_numbers(parser, x):
    try:
        res = int(x)
    except Exception as _:
        res = parser.parse(x)
        if not res:
            res = x  # If parse fails, it is probably a string
        else:
            res = int(res)  # If parse works, make sure an int is returned
    finally:
        return res


w2n = Word2Num()
df["response"] = df["response"].apply(lambda x: fix_word_numbers(w2n, x))

In [14]:
df.head()

,question_id,size_treatment,occluded_concepts,response
0,337814001,1,[28-table],3
1,337814001,1,[6-wall],3
2,337814001,1,[26-chair],3
3,337814001,1,[2-leg],3
4,337814001,1,[1-wall],3


Add column ```y``` representing the model answer
- If response is numeric, just use that
- If response is text, create lookup for the results of that question

In [15]:
lookup_y = {}

for idx, row in df.iterrows():
    q_idx = row["question_id"]
    if q_idx not in lookup_y:
        lookup_y[q_idx] = {}

    try:
        y = pd.to_numeric(row["response"])
        lookup_y[q_idx][row["response"]] = y
    except Exception as _:
        # Get all rows with same question ID
        curr_q = df[df["question_id"] == q_idx]
        # Create dict entry with list of unique responses
        lookup_y[q_idx] = {v: k for k, v in enumerate(curr_q["response"].unique())}
        y = lookup_y[q_idx][row["response"]]
    finally:
        df.loc[idx, "y"] = y

In [16]:
df.head()

,question_id,size_treatment,occluded_concepts,response,y
0,337814001,1,[28-table],3,3.0
1,337814001,1,[6-wall],3,3.0
2,337814001,1,[26-chair],3,3.0
3,337814001,1,[2-leg],3,3.0
4,337814001,1,[1-wall],3,3.0


In [17]:
# Make dictionary of dataframes w.r.t. question id
df_dict = {q_idx: df[df["question_id"] == q_idx] for q_idx in dict_counter_resp}

In [18]:
df_dict[337814001].head()

,question_id,size_treatment,occluded_concepts,response,y
0,337814001,1,[28-table],3,3.0
1,337814001,1,[6-wall],3,3.0
2,337814001,1,[26-chair],3,3.0
3,337814001,1,[2-leg],3,3.0
4,337814001,1,[1-wall],3,3.0


Let us make binary columns indicating whether a concept was occluded or not.
By default, all concepts are present (1). Then, we zero out the ones occluded.

Note: it is done at this point because each question has a different number of concepts.

In [19]:
# Adding 1 column (of zeros) to each sub-dataframe for each concept found
for q_idx in df_dict:
    for concept in dict_concepts[q_idx]:
        df_dict[q_idx][concept] = 1

In [20]:
df_dict[337814001].head()

,question_id,size_treatment,occluded_concepts,response,y,28-table,6-wall,26-chair,2-leg,1-wall,...,37-shoe,36-chair,15-head,25-arm,17-clock,18-shirt,20-pant,32-glass,5-face,14-leg
0,337814001,1,[28-table],3,3.0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
1,337814001,1,[6-wall],3,3.0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2,337814001,1,[26-chair],3,3.0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
3,337814001,1,[2-leg],3,3.0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
4,337814001,1,[1-wall],3,3.0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [21]:
# Correcting the dataframes based on occlusions
for q_idx in df_dict:
    for id, row in df_dict[q_idx].iterrows():
        curr_concepts = row["occluded_concepts"]
        for c in curr_concepts:
            df_dict[q_idx].loc[id, c] = 0
    df_dict[q_idx].drop(["occluded_concepts"], axis=1, inplace=True)

View the different responses from the models

In [22]:
# for q_idx in df_dict:
#     unique_res = df_dict[q_idx]["response"].unique()
#     print(f"Question {q_idx} -> {len(unique_res)} unique responses {unique_res}")

In [23]:
df_dict[337814001].head()

,question_id,size_treatment,response,y,28-table,6-wall,26-chair,2-leg,1-wall,35-man,37-shoe,36-chair,15-head,25-arm,17-clock,18-shirt,20-pant,32-glass,5-face,14-leg
0,337814001,1,3,3.0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
1,337814001,1,3,3.0,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2,337814001,1,3,3.0,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1
3,337814001,1,3,3.0,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1
4,337814001,1,3,3.0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1


```df_dict```
- Dictionary with question_ids as keys for which we have valid RKs
- Each entry in the dictionary is a dataframe with
  - The original response (no occlusion; size_treatment=0)
  - A bunch of rows with the counterfactual responses (combinations of concepts)
  - A series of binary columns matching 1:1 the concepts present or occluded

### Causal Analysis

First, we load the NX graphs and make the .dot string required by the dowhy package.

Make sure to run ```sk_processing/pickle_sk.py``` to obtain the pickled files first.

In [ ]:
# Load pickled SK graphs
sk_graphs = {}
all_img_ids = []
for ds_class in sk_hdl.get_classes():
    sk_hdl.set_curr_class(ds_class)
    curr_graphs = load_pickle(Path("..", sk_hdl.get_exp_pkl_path()))
    all_img_ids.extend(list(curr_graphs.keys()))
    print(f"Loaded {len(curr_graphs)} from {DATASET}+{ds_class}")
    print(f"- Old graph dict: {len(sk_graphs)} entries.")
    sk_graphs.update(curr_graphs)
    print(f"- New graph dict: {len(sk_graphs)} entries.")

Loaded 75 from vqav2+how_many_people_are
- Old graph dict: 0 entries.
- New graph dict: 75 entries.
Loaded 75 from vqav2+what_is_the_person
- Old graph dict: 75 entries.
- New graph dict: 148 entries.


In [25]:
print(f"Duplicate ids: {set([x for x in all_img_ids if all_img_ids.count(x) > 1])}")

Duplicate ids: {'154520', '473299'}


In [26]:
# Add to each graph a node corresponding to the outcome variable 'y'
for sk in sk_graphs.values():
    sk.add_node("y", label="response")

    for node in list(sk.nodes()):
        sk.add_edge(node, "y")

In [ ]:
# Load question data. Needed to match df_dict and sk_graphs
questions = []
for ds_class in ds_hdl.get_classes():
    ds_hdl.set_curr_class(ds_class)
    questions_path = Path("..", ds_hdl.get_sampled_questions_path())
    data = load_jsonl(questions_path)
    questions.extend(data)
print(f"Loaded {len(questions)} questions from {DATASET}.")

Loaded 150 questions from vqav2.


Running causal analysis

In [ ]:
def run_causal_inference(data, nx_graph, do_refute=False):
    all_estimates = []
    refutations = []

    # Prepare data for multinomial logistic regressor
    # responses = data["y"]  # 1-d vector of the responses
    cols_to_skip = ["question_id", "size_treatment", "response"]
    cols_to_keep = list(set(data.keys()) - set(cols_to_skip))
    data_for_estimation = data[cols_to_keep]  # n_observations x regressors
    regressors = list(data_for_estimation.drop("y", axis=1).keys())

    for col in regressors:
        print(f"Working on {col}...")

        # Exclude the current 'col' from the variables that can condition the estimates
        effect_modifiers = [r for r in regressors if r != col]

        # effect_modifiers: estimates are separately computed (conditioned)
        # for each value of the effect_modifier.
        model = CausalModel(
            data=data_for_estimation,
            treatment=col,
            outcome="y",
            graph=nx_graph,
            effect_modifiers=effect_modifiers,
            # missing_nodes_as_confounders=True,
        )
        estimand = model.identify_effect(proceed_when_unidentifiable=True)

        # Control and treatment values are correct since we
        # are occluding (removing) elements from images.
        ce_estimate = model.estimate_effect(
            estimand,
            # method_name="backdoor.generalized_linear_model",
            method_name="backdoor.linear_regression",
            control_value=1,
            treatment_value=0,
            test_significance=True,
            confidence_intervals=True,
            fit_estimator=True,
            # method_params={"glm_family": sm.families.Binomial()},
        )

        all_estimates.append(ce_estimate)

        # Run refutation with Placebo
        if do_refute:
            res_refutation = model.refute_estimate(
                estimand,
                ce_estimate,
                method_name="placebo_treatment_refuter",
                show_progress_bar=False,
                placebo_type="permute",
            )
            refutations.append({col: res_refutation})

    # Sort causal estimates before returning
    sorted_estimates = sorted(all_estimates, key=lambda x: x["value"], reverse=True)
    return sorted_estimates, refutations

In [127]:
def run_causal_inference_DML(
    data, nx_graph, test_significance=False, confidence_intervals=False, do_refute=False
):
    all_estimates = []
    refutations = []

    # Prepare data for multinomial logistic regressor
    # responses = data["y"]  # 1-d vector of the responses
    cols_to_skip = ["question_id", "size_treatment", "response"]
    cols_to_keep = list(set(data.keys()) - set(cols_to_skip))
    data_for_estimation = data[cols_to_keep]  # n_observations x regressors
    regressors = list(data_for_estimation.drop("y", axis=1).keys())

    for col in regressors:
        print(f"Working on {col}...")

        # Exclude the current 'col' from the variables that can condition the estimates
        effect_modifiers = [r for r in regressors if r != col]

        # effect_modifiers: estimates are separately computed (conditioned)
        # for each value of the effect_modifier.
        model = CausalModel(
            data=data_for_estimation,
            treatment=col,
            outcome="y",
            graph=nx_graph,
            effect_modifiers=effect_modifiers,
        )
        estimand = model.identify_effect(proceed_when_unidentifiable=True)

        # Define method params
        method_params = {
            "init_params": {
                "model_y": GradientBoostingRegressor(),
                "model_t": GradientBoostingRegressor(),
                "model_final": LassoCV(fit_intercept=False),
                "featurizer": PolynomialFeatures(degree=1, include_bias=False),
            },
            "fit_params": {},
        }
        if test_significance:
            method_params["fit_params"] = {
                "inference": BootstrapInference(n_bootstrap_samples=1, n_jobs=-1)
            }

        # Control and treatment values are correct since we
        # are occluding (removing) elements from images.
        ce_estimate = model.estimate_effect(
            estimand,
            method_name="backdoor.econml.dml.DML",
            control_value=1,
            treatment_value=0,
            test_significance=test_significance,
            confidence_intervals=confidence_intervals,
            fit_estimator=True,
            method_params=method_params,
        )

        all_estimates.append(ce_estimate)

        # Run refutation with Placebo
        if do_refute:
            res_refutation = model.refute_estimate(
                estimand,
                ce_estimate,
                method_name="placebo_treatment_refuter",
                show_progress_bar=False,
                placebo_type="permute",
            )
            refutations.append({col: res_refutation})

    # Sort causal estimates before returning
    sorted_estimates = sorted(all_estimates, key=lambda x: x.value, reverse=True)
    return sorted_estimates, refutations

In [128]:
estimates = {}
refutations = {}

for curr_q in questions:
    curr_idx = curr_q["question_id"]
    print(f">> Working with question # {curr_idx}")
    curr_sk_graph = copy.deepcopy(sk_graphs[curr_q["img"]])
    curr_df = df_dict[curr_idx]

    # removing self loops for causal analysis
    curr_sk_graph.remove_edges_from(nx.selfloop_edges(curr_sk_graph))

    # Save graph image
    # curr_out_dir = f"{CE_OUT_DIR}/{MODEL}/{DATASET}"
    # make_dir(curr_out_dir)
    # save_graph_to_img(g_idx, graph, curr_out_dir)

    # Compute causal effects
    estimates[curr_idx], refutations[curr_idx] = run_causal_inference_DML(
        curr_df,
        curr_sk_graph,
        test_significance=False,
        confidence_intervals=False,
        do_refute=False,
    )

    # Do computations only for 1, for now
    break

>> Working with question # 337814001
Working on 15-head...
Working on 6-wall...
Working on 20-pant...
Working on 17-clock...
Working on 2-leg...
Working on 28-table...
Working on 36-chair...
Working on 32-glass...
Working on 14-leg...
Working on 26-chair...
Working on 25-arm...
Working on 5-face...
Working on 35-man...
Working on 18-shirt...
Working on 37-shoe...
Working on 1-wall...


Save causal estimates to file

In [ ]:
causal_hdl = CausalHandler()
causal_hdl.set_curr_model(MODEL)
causal_hdl.set_curr_ds(DATASET)
estimates_out_path = base_dir.joinpath(Path(causal_hdl.get_estimates_path())).resolve()
make_dir(estimates_out_path.parent)

[Errno 17] File exists: '/Users/lorenzocorti/Developer/mllm_diagnosis/data/causal_effects/internvl2/vqav2'


In [139]:
import io
from contextlib import redirect_stdout

In [140]:
# Preparing data to save
def format_estimates(estimates, test_significance=False, std_error=False):
    out_data = {}
    for q_idx in estimates:
        if q_idx not in out_data:
            out_data[q_idx] = {}

        for curr_est in estimates[q_idx]:
            var_name = curr_est._treatment_name[0]
            if var_name not in out_data[q_idx]:
                out_data[q_idx][var_name] = {}

            to_save = {
                "estimate": curr_est.value,
                "estimand_type": curr_est.target_estimand.estimand_type.value,
                "estimand_expr": curr_est.realized_estimand_expr,
            }

            # Catch the output of the interpret() method
            f = io.StringIO()
            with redirect_stdout(f):
                curr_est.interpret()
            to_save["interpretation"] = f.getvalue()

            if test_significance:
                to_save["p-value"] = curr_est.test_stat_significance()
                # TODO: fix this
                to_save["significance"] = None

            if std_error:
                to_save["std_error"] = curr_est.get_standard_error()

            out_data[q_idx][var_name] = to_save
    return out_data


out_data = format_estimates(estimates, test_significance=False, std_error=False)

In [141]:
# save
save_json(out_data, estimates_out_path)